# Permutation Crossover Laboratory
Compare **ERX, CX, PMX, MX, OBX, PBX, UOX, OX, and LOX** under the same GA, seed, population, mutation, and evaluation budget. The central question is not simply *which crossover wins?* but *which representation matches the problem?*

In [ ]:
# Run once in a clean notebook environment.
%pip install -q git+https://github.com/WarinWatt/COINCIDENCE_algorithms_suite.git@agent/publish-coin-library pandas matplotlib

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.core.problem import Problem
from pymoo.optimize import minimize
from pymoo.operators.crossover.ox import OrderCrossover
from pymoo.operators.crossover.erx import EdgeRecombinationCrossover
from pymoo.operators.mutation.inversion import InversionMutation
from pymoo.operators.sampling.rnd import PermutationRandomSampling
from coin.adapters.pymoo.permutation_crossovers import PermutationCrossover

OPERATORS = ['erx', 'cx', 'pmx', 'mx', 'obx', 'pbx', 'uox', 'ox', 'lox']
REPRESENTATION = {
 'erx':'adjacent edges', 'cx':'absolute positions / cycles',
 'pmx':'position mapping', 'mx':'one-point prefix + relative order',
 'obx':'order among selected jobs', 'pbx':'fixed positions + remaining order',
 'uox':'uniform positions + remaining order', 'ox':'segment + cyclic order',
 'lox':'segment + linear order'}
pd.Series(REPRESENTATION, name='transmitted structure').to_frame()

In [ ]:
class PermutationFixture(Problem):
    def __init__(self, n, evaluate):
        super().__init__(n_var=n, n_obj=1, xl=0, xu=n-1, vtype=int)
        self.evaluate_permutation = evaluate
    def _evaluate(self, X, out, *args, **kwargs):
        out['F'] = np.asarray([self.evaluate_permutation(p) for p in X])[:, None]

def crossover(name, probability=.9):
    if name == 'ox': return OrderCrossover(prob=probability)
    if name == 'erx': return EdgeRecombinationCrossover(prob=probability)
    return PermutationCrossover(name, prob=probability)

def run_ga(name, fixture, seed=42, population=100, generations=400):
    algorithm = GA(pop_size=population, sampling=PermutationRandomSampling(),
        crossover=crossover(name), mutation=InversionMutation(prob=.2),
        eliminate_duplicates=True)
    started = time.perf_counter()
    result = minimize(fixture, algorithm, ('n_gen', generations), seed=seed,
                      save_history=True, verbose=False)
    return {'operator':name.upper(), 'best':float(result.F[0]),
            'runtime_s':time.perf_counter()-started,
            'history':[float(s.pop.get('F').min()) for s in result.history]}

In [ ]:
def classic_problem(kind, n=20, seed=42):
    rng = np.random.default_rng(seed)
    if kind == 'TSP':
        xy = rng.random((n,2))
        return PermutationFixture(n, lambda p: sum(np.linalg.norm(xy[p[i]]-xy[p[(i+1)%n]]) for i in range(n)))
    if kind == 'Flow Shop':
        times = rng.integers(1,100,(n,5))
        def makespan(p):
            c=np.zeros(5);
            for job in p:
                c[0]+=times[job,0]
                for m in range(1,5): c[m]=max(c[m],c[m-1])+times[job,m]
            return c[-1]
        return PermutationFixture(n,makespan)
    if kind == 'QAP':
        flow=rng.integers(0,20,(n,n)); distance=rng.integers(0,30,(n,n))
        flow=(flow+flow.T)//2; distance=(distance+distance.T)//2
        return PermutationFixture(n,lambda p: float((flow*distance[np.ix_(p,p)]).sum()))
    if kind == 'N-Queens':
        return PermutationFixture(n,lambda p: sum(abs(i-j)==abs(int(p[i])-int(p[j])) for i in range(n) for j in range(i+1,n)))
    if kind == 'Linear Ordering':
        weight=rng.integers(0,20,(n,n)); weight=np.triu(weight,1)
        def violations(p):
            pos=np.argsort(p); return sum(weight[i,j] for i in range(n) for j in range(i+1,n) if pos[i]>pos[j])
        return PermutationFixture(n,violations)
    side=int(np.ceil(np.sqrt(n))); xy=np.array([(i%side,i//side) for i in range(n)])
    def knight_loss(p):
        delta=np.abs(np.diff(xy[p],axis=0)); return int(sum(not tuple(d) in {(1,2),(2,1)} for d in delta))
    return PermutationFixture(n,knight_loss)

In [ ]:
# Teaching-sized comparison. Increase to population=100, generations=400 for the measured protocol.
problem_names=['TSP','Flow Shop','QAP','N-Queens','Linear Ordering',"Knight's Tour"]
records=[]
for problem_name in problem_names:
    fixture=classic_problem(problem_name,n=16,seed=42)
    for operator in OPERATORS:
        records.append({'problem':problem_name, **run_ga(operator,fixture,population=50,generations=100)})
results=pd.DataFrame(records)
results['rank']=results.groupby('problem')['best'].rank(method='min')
results.sort_values(['problem','rank'])[['problem','operator','best','rank','runtime_s']]

In [ ]:
pivot=results.pivot(index='operator',columns='problem',values='rank')
ax=pivot.plot.bar(figsize=(13,5),width=.85)
ax.set_ylabel('Rank (lower is better)'); ax.set_title('Crossover × problem structure')
plt.tight_layout(); plt.show()

## Exercises
1. Run at least 5 independent seeds and report paired ranks—not only the best run.
2. Compare OX and LOX: when does cyclic filling help a path that is not cyclic?
3. Compare PBX and UOX: does fixed inheritance reduce variance?
4. Plot convergence. ERX may still improve after order-based methods have stopped.
5. Replace synthetic fixtures with TSPLIB, QAPLIB, Taillard PFSP, or another declared benchmark.